# NeuroScan Nepal — Fine-tune on Google Colab (GPU)

1. **Runtime → Change runtime type → GPU (T4)**
2. **Runtime → Run all**
3. If files are missing on Drive, Colab will ask you to **Choose Files** — upload from your PC:
   - `NeuroScan_Nepal/scripts/neuroscan_data.zip`
   - `NeuroScan_Nepal/models/cnn_baseline.pth`

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU first: Runtime → Change runtime type → GPU (T4)'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q opencv-python-headless scikit-learn matplotlib

In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile
import shutil

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/final year project/NeuroScan_Nepal')
ZIP = BASE / 'neuroscan_data.zip'
PRETRAINED = BASE / 'cnn_baseline.pth'
SCRIPT = BASE / 'notebooks' / 'colab' / 'neuroscan_colab_train.py'
WORK = Path('/content/neuroscan')
OUT = Path('/content/output')
WORK.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

# Local copies (used if not on Drive)
LOCAL_ZIP = WORK / 'neuroscan_data.zip'
LOCAL_MODEL = OUT / 'cnn_baseline_pretrained.pth'

print('Project folder:', BASE.exists())
print('Dataset zip on Drive:', ZIP.exists())
print('Pretrained on Drive:', PRETRAINED.exists())
print('Train script on Drive:', SCRIPT.exists())

if not ZIP.exists():
    print('\n>>> Choose neuroscan_data.zip from your PC (scripts folder) <<<')
    up = files.upload()
    LOCAL_ZIP.write_bytes(up[next(iter(up))])
    zip_path = LOCAL_ZIP
else:
    zip_path = ZIP

if not PRETRAINED.exists():
    print('\n>>> Choose cnn_baseline.pth from your PC (models folder) <<<')
    up = files.upload()
    LOCAL_MODEL.write_bytes(up[next(iter(up))])
    pretrained_path = LOCAL_MODEL
else:
    pretrained_path = PRETRAINED

PRETRAINED = pretrained_path  # used by next cells

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(WORK)

DATA_ROOT = WORK if (WORK / 'normal').exists() else WORK / 'raw'
n = len(list((DATA_ROOT / 'normal').rglob('*.jpg')))
a = len(list((DATA_ROOT / 'abnormal').rglob('*.jpg')))
print(f'\nImages — Normal: {n}, Abnormal: {a}')
if n == 0 or a == 0:
    raise FileNotFoundError('No images in zip. Re-create: scripts/prepare_colab_upload.ps1 on PC')

if SCRIPT.exists():
    shutil.copy2(SCRIPT, Path('neuroscan_colab_train.py'))
    print('Training script copied from Drive')
else:
    print('Upload neuroscan_colab_train.py from notebooks/colab/ on your PC:')
    up = files.upload()
    Path('neuroscan_colab_train.py').write_bytes(up[next(iter(up))])

print('Ready to fine-tune. Pretrained:', PRETRAINED)

In [ ]:
# Fine-tune existing model (lower LR, fewer epochs)
!python neuroscan_colab_train.py \
  --data-root {DATA_ROOT} \
  --out-dir {OUT} \
  --pretrained {PRETRAINED} \
  --epochs 15 \
  --batch-size 64 \
  --lr 0.0001 \
  --patience 5

In [ ]:
from google.colab import files
import shutil

save = BASE / 'colab_finetune_output'
save.mkdir(exist_ok=True)

for f in OUT.iterdir():
    shutil.copy2(f, save / f.name)
    files.download(str(f))
    print('Downloaded:', f.name)

print('\nAlso saved on Drive:', save)
print('Copy cnn_baseline.pth + cnn_baseline_calibration.json to NeuroScan_Nepal/models/ on your PC')